# TriageIA — Fase 1: Exploración del Dataset

**Objetivo:** Analizar los 272 diálogos médico-paciente de `Dataset/text/text/` y tomar una decisión documentada sobre la representación textual a usar en el entrenamiento.

**Fuente:** Dataset OSCE/clínico (inglés), 272 casos, especialidades RES/MSK/GAS/CAR/DER/GEN.

**Salidas de este notebook:**
- `data/processed/dataset_exploracion_fase1.csv`
- `reports/figures/distribucion_grupos_clinicos.png`
- `docs/decisions/01_dataset_text_representation.md`

In [ ]:
import sys
import re
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
%matplotlib inline

# Añadir src/ al path para importar funciones del script
ROOT = Path().resolve().parent
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))

DATASET_DIR = ROOT / 'Dataset' / 'text' / 'text'
OUTPUT_CSV  = ROOT / 'data' / 'processed' / 'dataset_exploracion_fase1.csv'
OUTPUT_FIG  = ROOT / 'reports' / 'figures' / 'distribucion_grupos_clinicos.png'

print(f'ROOT: {ROOT}')
print(f'Dataset dir existe: {DATASET_DIR.exists()}')
print(f'CSV ya generado: {OUTPUT_CSV.exists()}')

## 1. Carga del CSV de exploración

El CSV fue generado por `src/eda_fase1.py`. Si no existe, ejecuta primero:
```
python src/eda_fase1.py
```

In [ ]:
df = pd.read_csv(OUTPUT_CSV)
print(f'Casos cargados: {len(df)}')
print(f'Columnas: {list(df.columns)}')
df.head(3)

## 2. Distribución por grupo clínico

El desbalanceo severo es el hallazgo clave de esta fase. **RES representa el 78% del dataset.**

In [ ]:
dist = df['grupo_clinico'].value_counts().sort_values(ascending=False)
total = len(df)

print('DISTRIBUCIÓN POR GRUPO CLÍNICO')
print('='*45)
for grupo, count in dist.items():
    bar = '█' * int(count / total * 40)
    print(f'  {grupo:<6} {count:>3} casos  ({count/total*100:5.1f}%)  {bar}')
print(f'  {'TOTAL':6} {total:>3} casos  (100.0%)')

In [ ]:
GROUP_COLORS = {'RES':'#4C72B0','MSK':'#55A868','GAS':'#C44E52',
                'CAR':'#8172B2','DER':'#CCB974','GEN':'#64B5CD'}

grupos = dist.index.tolist()
counts = dist.values.tolist()
colors = [GROUP_COLORS.get(g, '#999') for g in grupos]

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(grupos, counts, color=colors, edgecolor='white', linewidth=0.8)

for bar, count in zip(bars, counts):
    pct = count / total * 100
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1.5,
            f'{count}\n({pct:.1f}%)', ha='center', va='bottom',
            fontsize=10, fontweight='bold')

ax.set_title('Distribución de casos por grupo clínico\nDataset TriageIA — 272 diálogos médico-paciente',
             fontsize=13, pad=14)
ax.set_xlabel('Grupo clínico', fontsize=11)
ax.set_ylabel('Número de casos', fontsize=11)
ax.set_ylim(0, max(counts) * 1.20)
ax.yaxis.set_major_locator(mticker.MultipleLocator(20))
ax.spines[['top','right']].set_visible(False)
ax.grid(axis='y', alpha=0.3, linestyle='--')

fig.text(0.5, -0.04,
         'NOTA: Desbalanceo severo — RES representa ~78% del total.'
         ' Requerirá tratamiento en Fase 6-7 (class_weight o SMOTE).',
         ha='center', fontsize=8.5, color='gray', style='italic')

plt.tight_layout()
# ⚠️ CAPTURA PARA PRESENTACIÓN
fig.savefig(OUTPUT_FIG, dpi=150, bbox_inches='tight')
print(f'Figura guardada: {OUTPUT_FIG}')
plt.show()

## 3. Estadísticas de longitud

Analizamos cuánto texto aporta el paciente vs. el doctor, para fundamentar la decisión de representación.

In [ ]:
print('ESTADÍSTICAS DE LONGITUD (caracteres)')
print('='*65)
for col, label in [
    ('n_caracteres_total',    'Diálogo completo'),
    ('n_caracteres_paciente', 'Solo paciente   '),
    ('n_caracteres_doctor',   'Solo doctor     '),
]:
    s = df[col]
    print(f'  {label} | media: {s.mean():6.0f}  mediana: {s.median():6.0f}  '
          f'min: {s.min():5}  max: {s.max():5}')

print()
print('RATIO TEXTO PACIENTE / TOTAL (por grupo clínico)')
print('='*45)
df['ratio_paciente'] = df['n_caracteres_paciente'] / df['n_caracteres_total']
for grupo in dist.index:
    r = df[df['grupo_clinico'] == grupo]['ratio_paciente'].mean()
    print(f'  {grupo:<6}  {r:.1%}  de texto es del paciente')

In [ ]:
# Boxplot de longitud por grupo
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

grupos_presentes = dist.index.tolist()
data_total = [df[df['grupo_clinico']==g]['n_caracteres_total'].values for g in grupos_presentes]
data_pac   = [df[df['grupo_clinico']==g]['n_caracteres_paciente'].values for g in grupos_presentes]

bp1 = axes[0].boxplot(data_total, labels=grupos_presentes, patch_artist=True)
for patch, g in zip(bp1['boxes'], grupos_presentes):
    patch.set_facecolor(GROUP_COLORS.get(g, '#999'))
axes[0].set_title('Longitud diálogo completo (chars)', fontsize=11)
axes[0].set_xlabel('Grupo clínico')
axes[0].set_ylabel('Caracteres')
axes[0].grid(axis='y', alpha=0.3)

bp2 = axes[1].boxplot(data_pac, labels=grupos_presentes, patch_artist=True)
for patch, g in zip(bp2['boxes'], grupos_presentes):
    patch.set_facecolor(GROUP_COLORS.get(g, '#999'))
axes[1].set_title('Longitud texto paciente (chars)', fontsize=11)
axes[1].set_xlabel('Grupo clínico')
axes[1].set_ylabel('Caracteres')
axes[1].grid(axis='y', alpha=0.3)

plt.suptitle('Distribución de longitud por grupo — Comparativa D: completo vs solo P:', fontsize=12)
plt.tight_layout()
plt.show()

## 4. Análisis de calidad

In [ ]:
print('RESUMEN DE CALIDAD DEL DATASET')
print('='*50)
print(f'  Total de archivos:          {len(df)}')
print(f'  Sin problemas (OK):         {(df["observaciones_calidad"]=="OK").sum()}')
print(f'  Con observaciones:          {(df["observaciones_calidad"]!="OK").sum()}')

problemas = df[df['observaciones_calidad'] != 'OK'][['id_caso','grupo_clinico','observaciones_calidad']]
if not problemas.empty:
    print()
    print('Archivos con observaciones:')
    print(problemas.to_string(index=False))
    print()
    print('NOTAS:')
    print('  FORMATO_D_PUNTO_COMA: El archivo usa D; en lugar de D:')
    print('    → Typo tipográfico. El parser lo maneja. No afecta al dataset.')
    print('  RES0002 / RES0054: Archivos UTF-16 LE (con BOM).')
    print('    → El parser los detecta y decodifica correctamente.')

## 5. Ejemplos representativos por grupo

In [ ]:
print('EJEMPLOS — PRIMERAS 3 LÍNEAS DEL TEXTO PACIENTE POR GRUPO')
print('='*70)
for grupo in dist.index:
    ejemplo = df[df['grupo_clinico'] == grupo].iloc[0]
    texto_muestra = ejemplo['texto_paciente'][:300].replace('\n', ' ')
    print(f'\n[{grupo}] {ejemplo["id_caso"]}')
    print(f'  {texto_muestra}...')

## 6. Decisión de representación textual

### Alternativas evaluadas

| Opción | Ventaja | Riesgo |
|--------|---------|--------|
| Solo `texto_paciente` | Elimina ruido del doctor | Mismatch con Whisper (no separa hablantes) |
| `texto_dialogo_completo` | Coherente con inferencia real | Potencial leakage por preguntas del doctor |
| Turno inicial P: | Chief complaint puro | Pierde información de respuestas posteriores |
| NER → síntomas | Representación semántica pura | Requiere pipeline Fase 4 (no disponible aún) |

### Decisión adoptada

**Enfoque híbrido escalonado:**
1. `texto_dialogo_completo` como **baseline principal** (Fase 7)
2. `texto_paciente` como **alternativa** a comparar en Fase 7
3. `texto_doctor` conservado para análisis, no para entrenamiento

**Justificación clave:** Whisper no aplica diarización. La app final recibirá el audio de toda la consulta transcrito en bruto. Entrenar con el diálogo completo reproduce esa condición de inferencia.

**Documento de decisión:** `docs/decisions/01_dataset_text_representation.md`

In [ ]:
# Resumen final
print('RESUMEN FASE 1')
print('='*50)
print(f'  Casos totales:              {len(df)}')
print(f'  Grupos clínicos:            {df["grupo_clinico"].nunique()}')
print(f'  Grupo mayoritario:          RES ({dist["RES"]} casos, {dist["RES"]/total*100:.1f}%)')
print(f'  Longitud media (completo):  {df["n_caracteres_total"].mean():.0f} chars')
print(f'  Longitud media (paciente):  {df["n_caracteres_paciente"].mean():.0f} chars')
print(f'  Ratio paciente/total:       {df["ratio_paciente"].mean():.1%}')
print(f'  Turnos P: medios:           {df["n_turnos_paciente"].mean():.1f}')
print(f'  Archivos con problemas:     {(df["observaciones_calidad"]!="OK").sum()} (solo formato, no datos)')
print()
print('PENDIENTE PARA FASE 2:')
print('  → Asignar niveles Manchester (estrategia a decidir)')
print('  → Construir dataset_maestro.csv con columna nivel_manchester')
print('  → Representación principal: texto_dialogo_completo')